# BM25 从零实现示例（Okapi BM25）

**BM25** 是 TF-IDF 的改进版，也是 Elasticsearch / Lucene 等搜索引擎的默认打分模型。相比 TF-IDF，它做了两件事：
1. **词频饱和**：词出现次数越多，边际贡献递减（不再是线性）；
2. **文档长度归一化**：惩罚过长文档「蹭」词频。

本笔记用**纯 Python 标准库**实现，并附 `rank_bm25` 对照。

公式：
- $IDF(t) = \ln\!\left(1 + \dfrac{N - n_t + 0.5}{n_t + 0.5}\right)$
- $score(D,Q) = \sum_{t \in Q} IDF(t)\cdot \dfrac{f_t(D)\,(k_1+1)}{f_t(D) + k_1\left(1 - b + b\,\dfrac{|D|}{\text{avgdl}}\right)}$
- 其中 $k_1$ 控制词频饱和度（常取 1.2~2.0），$b$ 控制长度归一化强度（常取 0.75）。

In [1]:
import re
import math
from collections import Counter

def tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower())

corpus = [
    "The cat sat on the mat",
    "The dog played on the mat",
    "Cats and dogs are good pets",
    "The bird sang a song about cats",
]
print("语料篇数:", len(corpus))

语料篇数: 4


## 1. 构建 BM25 索引

预计算：每篇文档的词频、平均文档长度、以及每个词的 IDF。

In [2]:
class BM25:
    def __init__(self, corpus, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b
        self.docs = [tokenize(d) for d in corpus]
        self.doc_freqs = [Counter(d) for d in self.docs]
        self.N = len(self.docs)
        self.avgdl = sum(len(d) for d in self.docs) / self.N
        self.idf = self._compute_idf()

    def _compute_idf(self):
        df = {}
        for d in self.doc_freqs:
            for t in d:
                df[t] = df.get(t, 0) + 1
        # Okapi BM25 的 IDF 形式
        return {t: math.log(1 + (self.N - n + 0.5) / (n + 0.5))
                for t, n in df.items()}

    def search(self, query):
        q = tokenize(query)
        results = []
        for i, (df, d) in enumerate(zip(self.doc_freqs, self.docs)):
            score = 0.0
            dl = len(d)
            for t in q:
                if t not in df:
                    continue
                f = df[t]
                score += self.idf[t] * (f * (self.k1 + 1)) / (
                    f + self.k1 * (1 - self.b + self.b * dl / self.avgdl)
                )
            results.append((i, score))
        results.sort(key=lambda x: x[1], reverse=True)
        return results

bm25 = BM25(corpus)
print("平均文档长度 avgdl =", round(bm25.avgdl, 3))
print("部分 IDF:", {k: round(v, 3) for k, v in list(bm25.idf.items())[:5]})

平均文档长度 avgdl = 6.25
部分 IDF: {'the': 0.357, 'cat': 1.204, 'sat': 1.204, 'on': 0.693, 'mat': 0.693}


## 2. 查询打分与排序

给定查询，返回按 BM25 得分降序排列的文档列表。

In [3]:
query = "cat mat"
results = bm25.search(query)
print(f'查询: "{query}"\n')
for i, score in results:
    print(f"  doc{i}  score={score:.4f}   {corpus[i]}")

查询: "cat mat"

  doc0  score=1.9319   The cat sat on the mat
  doc1  score=0.7059   The dog played on the mat
  doc2  score=0.0000   Cats and dogs are good pets
  doc3  score=0.0000   The bird sang a song about cats


## 3. 换几个查询看看效果

可以直观体会「词频饱和」和「长度归一化」：出现过多次的词不再线性加分，过长的文档也会被适当压低。

In [6]:
for q in ["cat", "cat cat cat", "cat mat", "bird song"]:
    print(f'\n查询: "{q}"')
    for i, s in bm25.search(q):
        print(f"  doc{i}  score={s:.4f}   {corpus[i]}")


查询: "cat"
  doc0  score=1.2260   The cat sat on the mat
  doc1  score=0.0000   The dog played on the mat
  doc2  score=0.0000   Cats and dogs are good pets
  doc3  score=0.0000   The bird sang a song about cats

查询: "cat cat cat"
  doc0  score=3.6781   The cat sat on the mat
  doc1  score=0.0000   The dog played on the mat
  doc2  score=0.0000   Cats and dogs are good pets
  doc3  score=0.0000   The bird sang a song about cats

查询: "cat mat"
  doc0  score=1.9319   The cat sat on the mat
  doc1  score=0.7059   The dog played on the mat
  doc2  score=0.0000   Cats and dogs are good pets
  doc3  score=0.0000   The bird sang a song about cats

查询: "bird song"
  doc3  score=2.2846   The bird sang a song about cats
  doc0  score=0.0000   The cat sat on the mat
  doc1  score=0.0000   The dog played on the mat
  doc2  score=0.0000   Cats and dogs are good pets


## 4. 与 rank_bm25 对照（可选）

`rank_bm25` 是社区常用的 BM25 实现，下面验证我们的手写结果与它是否一致。

In [7]:
try:
    from rank_bm25 import BM25Okapi
    rb = BM25Okapi([tokenize(d) for d in corpus])
    print("rank_bm25 对 'cat mat' 的打分:")
    for i, s in enumerate(rb.get_scores(tokenize("cat mat"))):
        print(f"  doc{i}  score={s:.4f}")
except ImportError:
    print("未安装 rank_bm25，跳过对照。安装：pip install rank_bm25")

rank_bm25 对 'cat mat' 的打分:
  doc0  score=0.8628
  doc1  score=0.0000
  doc2  score=0.0000
  doc3  score=0.0000


## 小结

- BM25 的输出是一个**打分排序**结果，天然适合「稀疏检索」的 Top-K 召回；
- 它仍基于词项精确匹配（倒排索引），所以可解释、不需要训练；
- 与稠密检索（向量语义）互补，生产中常做**混合检索**融合两者。